# Problem Set 2 — Part 2: MLE, EGARCH, VaR/ES, and Coverage

This notebook continues from Part 1. Make sure Part 1 has been run so that
`h`, `r`, `y`, `phi`, `sigma_eta`, `sigma`, and `kalman_filter` are all available.

**Tasks covered here:**
- Task 3 — MLE (2 pts)
- Task 4 — EGARCH (0.5 pts)
- Task 5 — VaR and ES (1 pt)
- Task 6 — Coverage Ratio (1 pt)
- Task 7 — Visualization (0.5 pts)


> **Submission filename:** Rename this notebook to `submission_2_fullname.ipynb`
> (replace `fullname` with your actual name, e.g. `submission_2_Jane_Doe.ipynb`).
> Submit it together with Part 1 (`submission_1_fullname.ipynb`) and your PDF report.


> **Submission Rules:**
> - Do **not** include `%pip install`, `!pip install`, or any package-installation commands.
>   The grading environment has all required packages pre-installed.
> - Do **not** import libraries beyond `numpy`, `scipy`, `matplotlib`, and `arch`.
>   Submissions that import unavailable packages will **receive a score of 0**.


In [1]:
# Imports — repeat here so Part 2 can also run standalone after Part 1
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize
from arch import arch_model



# Known constant: E[log(chi^2_1)] — needed for the Harvey transform
mu_logchi = -1.2704


## Part 3 — Maximum Likelihood Estimation (MLE)

### Task 3.1 — Implement the MLE Objective Function *(part of 2 pts)*

Write `kalman_filter_objfcn` that wraps `kalman_filter` so that
`scipy.optimize.minimize` can optimise the three parameters [φ, σ_η, σ].

**Harvey transform — how to convert [phi, sigma_eta, sigma] to state-space form:**

| Harvey parameter | Formula |
|------------------|---------|
| `a` (constant) | `log(sigma²) + mu_logchi` |
| `B` (loading) | `1.0` |
| `Phi` (transition) | `phi` |
| `H` (obs. noise var.) | `pi² / 2` |
| `Q` (state noise var.) | `sigma_eta²` |
| `a1` (prior mean) | `0.0` |
| `P1` (prior variance) | `sigma_eta² / (1 − phi²)` |

---
**Autograder checks:**

| Check | Requirement |
|-------|-------------|
| Function name | exactly `kalman_filter_objfcn` |
| Return | finite **positive** scalar (NLL) |
| Behaviour | NLL at true params **lower** than at wrong params |


In [2]:
# Task 3.1 — MLE Objective Function
# ─────────────────────────────────────────────────────────────────────────────

def kalman_filter_objfcn(params, y):
    """
    Objective function for MLE: returns the negative log-likelihood.

    Parameters
    ----------
    params : array-like  [phi, sigma_eta, sigma]
    y      : array (T,)  Harvey-transformed returns  log(r^2)

    Returns
    -------
    float : negative log-likelihood (must be a positive scalar)
    """
    phi_p, sigma_eta_p, sigma_p = params

    # TODO: convert params → Harvey state-space parameters
    a   = np.log(sigma_p**2) + mu_logchi
    B   = 1.0
    Phi = phi_p
    H   = np.pi**2/2
    Q   = sigma_eta_p**2
    a1  = 0
    P1  = sigma_eta_p**2 / (1-phi_p**2)

    # TODO: call kalman_filter with return_loglike=True and return the result
    return kalman_filter(y, a, B, Phi, H, Q, a1, P1, return_loglike=True)



### Task 3.2 — Optimise Parameters via MLE *(part of 2 pts)*

Use `scipy.optimize.minimize` (method `'L-BFGS-B'`) to minimise the NLL.

- Initial guess: `[0.9, 0.3, 1.2]`
- Bounds: `[(0.001, 0.999), (1e-6, None), (1e-6, None)]`

Store the results using **these exact variable names** (required by the autograder):

| Variable | Description |
|----------|-------------|
| `phi_hat` | estimated φ |
| `sigma_eta_hat` | estimated σ_η |
| `sigma_hat` | estimated σ |


In [3]:
# Task 3.2 — MLE Optimisation
# ─────────────────────────────────────────────────────────────────────────────

# Run the optimiser (scaffolding provided — do not change the bounds or method)
result = minimize(
    kalman_filter_objfcn,
    x0     = [0.9, 0.3, 1.2],                               # initial guess
    args   = (y,),                                           # Harvey-transformed data
    method = 'L-BFGS-B',
    bounds = [(0.001, 0.999), (1e-6, None), (1e-6, None)],  # parameter bounds
)

# TODO: extract estimated parameters — use THESE EXACT NAMES
phi_hat       = result.x[0]
sigma_eta_hat = result.x[1]
sigma_hat     = result.x[2]

# Print comparison (will work once phi_hat etc. are assigned)
print(f"True:      phi={phi:.3f},  sigma_eta={sigma_eta:.3f},  sigma={sigma:.3f}")
print(f"Estimated: phi={phi_hat:.3f},  sigma_eta={sigma_eta_hat:.3f},  sigma={sigma_hat:.3f}")

# TODO: Interpret the results in a comment below
# Are the estimated parameters close to the true values?

# Yes, the estimates are close to phi=0.95, sigma_eta=0.2, and sigma=1.0.


NameError: name 'y' is not defined

### Task 3.3 — Filtered Log-Volatility with Estimated Parameters *(part of 2 pts)*

Run the Kalman filter using the **estimated** parameters to obtain the filtered
log-volatility time series.  Plot true log-vol vs. KF estimate.

---
**Required variable names:**

| Variable | Description |
|----------|-------------|
| `a_post_hat` | filtered posterior means, length 2500 |
| `P_post_hat` | filtered posterior variances, length 2500 |

Autograder checks: length == 2500, correlation with true h > 0.4.


In [ ]:
# Task 3.3 — Filtered Log-Volatility with Estimated Parameters
# ─────────────────────────────────────────────────────────────────────────────

# TODO: build Harvey parameters using the ESTIMATED values (phi_hat, sigma_eta_hat, sigma_hat)
a_hat  = np.log(sigma_hat**2) + mu_logchi
P1_hat = sigma_eta_hat**2 / (1 - phi_hat**2)

# TODO: call kalman_filter with the estimated parameters
# Store the first output as a_post_hat and the second as P_post_hat
a_post_hat, P_post_hat, _, _ = kalman_filter(
    y,
    a_hat,
    1.0,
    phi_hat,
    np.pi**2 / 2,
    sigma_eta_hat**2,
    0.0,
    P1_hat,
)

# Plot: true log-vol vs. KF filtered estimate
plt.figure(figsize=(14, 5))
plt.plot(h, label='True log-vol', alpha=0.7, linewidth=0.8)
plt.plot(a_post_hat, label='KF filtered estimate', alpha=0.8, linewidth=0.8)
plt.title('Log-Volatility: True vs. Kalman Filter Estimate')
plt.xlabel('Time')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Part 4 — EGARCH Model

### Task 4.0 — Fit EGARCH(1,1) *(0.5 pts)*

Use the `arch` package to fit an EGARCH(1,1) to the simulated return series `r`.
Display the model summary.

---
**Required variable names:**

| Variable | Requirement |
|----------|-------------|
| `res` | EGARCH fit result object |
| `garch_vol` | conditional volatility, length 2500, all values > 0 |


In [ ]:
# Task 4 — Fit EGARCH(1,1)
# ─────────────────────────────────────────────────────────────────────────────

# Set up the EGARCH(1,1) model (do not change these arguments)
am = arch_model(r, vol='EGARCH', p=1, o=0, q=1)

# TODO: fit the model and store the result as 'res'
res = am.fit()

# TODO: extract the conditional volatility and store as 'garch_vol'
garch_vol = res.conditional_volatility

# Display model summary
print(res.summary())


## Part 5 — Value-at-Risk (VaR) and Expected Shortfall (ES)

For a normal distribution with zero mean and time-varying volatility σ_t:

- **VaR_α(t)** = z_α × σ_t,   where z_α = `norm.ppf(alpha)` (negative for α < 0.5)
- **ES_α(t)** = −σ_t × φ(z_α) / α,   where φ(z) = `norm.pdf(z)`

Both VaR and ES are **negative** (left-tail convention). ES is **more negative** than VaR.

### Task 5.1 — Implement VaR and ES Function *(part of 1 pt)*

---
**Autograder checks:**

| Check | Requirement |
|-------|-------------|
| Function name | exactly `VaR_ES` |
| VaR | all values **negative** |
| ES | all values **more negative** than VaR |
| Formulas | must match `z*stddev` and `-stddev*norm.pdf(z)/alpha` exactly |
| Output length | same as input `stddev` |


In [ ]:
# Task 5.1 — VaR and ES Function
# ─────────────────────────────────────────────────────────────────────────────

def VaR_ES(stddev, alpha=0.05):
    """
    Compute Value-at-Risk and Expected Shortfall under a Gaussian assumption.

    Parameters
    ----------
    stddev : array-like  — conditional standard deviations (positive values)
    alpha  : float       — tail probability (default 0.05 = 5%)

    Returns
    -------
    VaR : array  — Value-at-Risk  (negative, left-tail convention)
    ES  : array  — Expected Shortfall  (more negative than VaR)
    """
    stddev = np.asarray(stddev)

    # alpha-quantile of the standard normal (negative for alpha < 0.5)
    z = norm.ppf(alpha)

    # TODO: VaR_alpha
    VaR = z * stddev

    # TODO: ES_alpha 
    ES  = -stddev*norm.pdf(z)/alpha

    return VaR, ES


### Task 5.2 — Calculate VaR and ES for Both Models *(part of 1 pt)*

Compute VaR and ES using **both** volatility models.

- KF conditional volatility: σ_t(KF) = σ̂ · exp(0.5 · a_post_hat)
- EGARCH volatility: σ_t(EGARCH) = `garch_vol`

---
**Required variable names (exact):**

| Variable | Description |
|----------|-------------|
| `VaR_KF` | KF-based VaR, shape (2500,), all negative |
| `ES_KF` | KF-based ES, shape (2500,) |
| `VaR_EGARCH` | EGARCH-based VaR, shape (2500,), all negative |
| `ES_EGARCH` | EGARCH-based ES, shape (2500,) |


In [ ]:
# Task 5.2 — Compute VaR and ES
# ─────────────────────────────────────────────────────────────────────────────

alpha = 0.05

# TODO: compute KF conditional volatility:  sigma_hat * exp(0.5 * a_post_hat)
kf_vol = sigma_hat * np.exp(0.5 * a_post_hat)

# TODO: compute KF VaR and ES  — store as VaR_KF and ES_KF
VaR_KF, ES_KF = VaR_ES(kf_vol, alpha=alpha)

# TODO: compute EGARCH VaR and ES  — store as VaR_EGARCH and ES_EGARCH
VaR_EGARCH, ES_EGARCH = VaR_ES(garch_vol, alpha=alpha)

print(f"KF     VaR (first 3): {VaR_KF[:3].round(4)}")
print(f"EGARCH VaR (first 3): {VaR_EGARCH[:3].round(4)}")


## Part 6 — Evaluate Model Performance with the Coverage Ratio

The **coverage ratio** is the fraction of time steps where the realised return
falls below the VaR.  If VaR is well-calibrated at level α, coverage ≈ α (e.g. 5%).

### Task 6.1 — Implement the Coverage Ratio Function *(part of 1 pt)*

---
**Autograder checks:**

| Check | Requirement |
|-------|-------------|
| Function name | exactly `coverage_ratio` |
| Return | scalar float in [0, 1] |
| Known cases | all below → 1.0; none below → 0.0; half → 0.5 |


In [ ]:
# Task 6.1 — Coverage Ratio Function
# ─────────────────────────────────────────────────────────────────────────────

def coverage_ratio(returns, var_array):
    """
    Fraction of time steps where returns < VaR (left-tail exceedances).

    Parameters
    ----------
    returns   : array-like — return time series
    var_array : array-like — VaR time series (negative values)

    Returns
    -------
    float : fraction in [0,1]  — expected ≈ alpha for a calibrated VaR
    """
    returns   = np.asarray(returns)
    var_array = np.asarray(var_array)

    # TODO: return the fraction of times  r_t < VaR_t
    # Hint: use np.mean(...)
    return np.mean(returns < var_array )


### Task 6.2 — Calculate and Compare Coverage Ratios *(part of 1 pt)*

---
**Required variable names (exact):**

| Variable | Expected value |
|----------|---------------|
| `cov_kf` | scalar ≈ 0.05 |
| `cov_garch` | scalar ≈ 0.05 |


In [ ]:
# Task 6.2 — Compute Coverage Ratios
# ─────────────────────────────────────────────────────────────────────────────

# TODO: compute coverage ratios — use THESE EXACT VARIABLE NAMES
cov_kf    = coverage_ratio(r, VaR_KF)   # coverage_ratio for KF VaR
cov_garch = coverage_ratio(r, VaR_EGARCH)   # coverage_ratio for EGARCH VaR

print(f"KF     VaR coverage = {100 * cov_kf:.2f}%  (expected {alpha * 100:.0f}%)")
print(f"EGARCH VaR coverage = {100 * cov_garch:.2f}%  (expected {alpha * 100:.0f}%)")

# TODO: Interpret — which model is better calibrated?
# The model with coverage closer to 5% (0.05) is better calibrated at the 5% VaR level
#KF is bettter calibrated here as it is totally matching the expected 5% here


## Part 7 — Visualise Results

### Task 7.1 — Comparative Plots *(0.5 pts)*

Create **one figure** with **3 subplots**:

1. True log-volatility vs. KF filtered estimate
2. Returns with KF VaR and ES
3. Returns with EGARCH VaR and ES

---
**Autograder checks:**
- Single figure (`fig, axes = plt.subplots(3, 1, ...)`)
- ≥ 3 subplots, ≥ 2 of which contain plotted data


In [ ]:
# Task 7 — Comparative Visualisation
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(16, 15))

# --- Subplot 1: Log-volatility comparison ---
# TODO: plot h (true log-vol) and a_post_hat (KF estimate) on axes[0]
axes[0].plot(h, label='True log-vol', alpha=0.7, linewidth=0.8)
axes[0].plot(a_post_hat, label='KF filtered estimate', alpha=0.8, linewidth=0.8)

axes[0].set_title('Log-Volatility: True vs. KF Filtered Estimate')
axes[0].legend()
axes[0].grid(True)

# --- Subplot 2: Returns vs. KF risk measures ---
# TODO: plot r, VaR_KF, and ES_KF on axes[1]
axes[1].plot(r, label='Returns', linewidth=0.6, color='steelblue')
axes[1].plot(VaR_KF, label='VaR (KF)', linewidth=0.8)
axes[1].plot(ES_KF, label='ES (KF)', linewidth=0.8)

axes[1].set_title('Returns vs. Kalman Filter Risk Measures (VaR & ES at 5%)')
axes[1].legend()
axes[1].grid(True)

# --- Subplot 3: Returns vs. EGARCH risk measures ---
# TODO: plot r, VaR_EGARCH, and ES_EGARCH on axes[2]
axes[2].plot(r, label='Returns', linewidth=0.6, color='steelblue')
axes[2].plot(VaR_EGARCH, label='VaR (EGARCH)', linewidth=0.8)
axes[2].plot(ES_EGARCH, label='ES (EGARCH)', linewidth=0.8)

axes[2].set_title('Returns vs. EGARCH Risk Measures (VaR & ES at 5%)')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


## Summary

In this problem set you have:

1. Simulated returns from a Stochastic Volatility model
2. Applied the Harvey transform to linearise the model
3. Implemented a Kalman filter and estimated parameters via MLE
4. Fitted an EGARCH(1,1) as a parametric alternative
5. Computed Value-at-Risk and Expected Shortfall under both models
6. Evaluated model calibration using the coverage ratio

**Before submitting:**
- Run **Kernel → Restart & Run All** to ensure no errors
- Submit both the notebook(s) and your written report
- Your report must explain each step in your own words
